In [1]:
import torch
import torch.nn as nn 
import torch.optim as optim

In [3]:
import torchvision
from torchvision.datasets import MNIST
from torch.utils.data import Dataset , DataLoader
from torchvision.transforms import transforms

In [4]:
transform=transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.1307),(0.3081))
    ]
)

In [8]:
trainset=MNIST(root="./data",transform=transform,train=True)
testset=MNIST(root="./data",transform=transform,train=False)

In [9]:
trainset

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.1307, std=0.3081)
           )

In [10]:
testset

Dataset MNIST
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.1307, std=0.3081)
           )

In [11]:
trainloader=DataLoader(trainset,batch_size=32,shuffle=True)
testloader=DataLoader(testset,batch_size=32)

### RNN

In [24]:
class RNN(nn.Module):
    def __init__(self,input_size=28,hidden_size=128,num_layers=1,num_classes=10):
        super().__init__()

        self.hidden_size=hidden_size
        self.num_layers=num_layers

        self.rnn=nn.RNN(
            input_size,hidden_size,num_layers,batch_first=True
        )

        self.fc_layers=nn.Linear(hidden_size,num_classes)

    def forward(self,x):

        x=x.squeeze(1)

        h0=torch.zeros(
            self.num_layers,
            x.size(0),
            self.hidden_size
        )

        out,_=self.rnn(x,h0)

        out=out[:,-1,:]

        output=self.fc_layers(out)

        return output

In [25]:
model=RNN()

In [28]:
criteria=nn.CrossEntropyLoss()

optimizer=optim.Adam(model.parameters())

In [29]:
model.train()

epochs=10

for epoch in range(epochs):

    training_loss=0.0

    for xb,yb in trainloader:

        optimizer.zero_grad()

        outputs=model(xb)

        loss=criteria(outputs,yb)

        loss.backward()

        optimizer.step()

        training_loss+=loss.item()

    print(f"{epoch+1}/{epochs} => loss is {training_loss/len(trainloader)}")
    

1/10 => loss is 0.6551108108401299
2/10 => loss is 0.27385492937266825
3/10 => loss is 0.223345189439257
4/10 => loss is 0.19538423663079738
5/10 => loss is 0.17390389452576638
6/10 => loss is 0.16653503742069006
7/10 => loss is 0.16889039849564433
8/10 => loss is 0.1523813242971897
9/10 => loss is 0.13876279763281346
10/10 => loss is 0.15323499377071859


In [30]:
model.eval()

correct=0
total=0

with torch.no_grad():
    for xb,yb in testloader:

        output=model(xb)

        _,predicted=torch.max(output,1)

        correct=(predicted==yb).sum().item()

        total=yb.size(0)

        accuracy=correct/total*100

    print(accuracy)

100.0


### CNN for Digit Classification

In [37]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers=nn.Sequential(
            nn.Conv2d(1,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )

        self.fc_layers=nn.Sequential(
            nn.Linear(128*3*3,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)

        return x

In [38]:
model=CNN()

In [39]:
loss=nn.CrossEntropyLoss()

In [40]:
optimizer=optim.Adam(model.parameters())

In [41]:
epochs=10

model.train()

for epoch in range(epochs):

    training_loss=0.0

    for xb,yb in trainloader:

        optimizer.zero_grad()

        outputs=model(xb)

        loss=criteria(outputs,yb)

        loss.backward()

        optimizer.step()

        training_loss+=loss.item()

    print(f"{epoch+1}/{epochs} => loss is {training_loss/len(trainloader)}")

1/10 => loss is 0.12274290266255848
2/10 => loss is 0.04110035263290435
3/10 => loss is 0.02887911732223826
4/10 => loss is 0.021237895422882867
5/10 => loss is 0.01762297028778727
6/10 => loss is 0.01557339484165071
7/10 => loss is 0.012320520282472672
8/10 => loss is 0.012667125569896265
9/10 => loss is 0.008966836782246454
10/10 => loss is 0.009443718563273371


In [42]:
model.eval()

correct=0
total=0

with torch.no_grad():

    for xb,yb in testloader:

        output=model(xb)

        _,predicted=torch.max(output,1)

        correct+=(predicted==yb).sum().item()

        total+=yb.size(0)

        accuracy=correct/total*100

    print(accuracy)

99.05000000000001
